<a href="https://colab.research.google.com/github/zelal-Eizaldeen/deeplearning_course/blob/main/tensorflow_word_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- In this programming example, we will show how **to train a language model and embeddings jointly**, and then explore the embeddings a little bit in **TensorFlow**.

 In this notebook, we are **creating a language model and training it**, and it uses **embedding layers**. And then we are going to explore those embedding layers in TensorFlow.

So we **start with our usual imports.**. We're also going to use a **tokenizer** that will t**ake each word and make a token out of it**. We are going to train this on the book "Frankenstein".  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


The text for Mary Shelley's Frankenstein can be downloaded from https://www.gutenberg.org/files/84/84-0.txt. Rename the file to frankenstein.txt and copy to the data directory.

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text \
    import text_to_word_sequence
import tensorflow as tf
import logging
tf.get_logger().setLevel(logging.ERROR)

EPOCHS = 32
BATCH_SIZE = 256
INPUT_FILE_NAME = '../data/frankenstein.txt'
WINDOW_LENGTH = 40
WINDOW_STEP = 3
PREDICT_LENGTH = 3
MAX_WORDS = 7500
EMBEDDING_WIDTH = 100

we are going to open the file, read the file,

In [ ]:
# Open the input file.
file = open(INPUT_FILE_NAME, 'r', encoding='utf-8-sig')
text = file.read()
file.close()

we're going to split this file into individual words. So we use this convenience method, **text to word sequence**.

In [ ]:
# Make lower case and split into individual words.
text = text_to_word_sequence(text)

we're gonna create this  **training fragment**, of individual words, So we do that and

In [ ]:
# Create training examples.
fragments = []
targets = []
for i in range(0, len(text) - WINDOW_LENGTH, WINDOW_STEP):
    fragments.append(text[i: i + WINDOW_LENGTH])
    targets.append(text[i + WINDOW_LENGTH])

then we are goint to invoke this tokenizer that takes each word and convert it to an index.

And we also have a special token here for words that are **out of the vocabulary**, because we decided we will at most have max words that we're working with. So this max words is *7,500. That's the number of unique words that we'll be using*.
- if we then try to **use some other word**, it's going to **be mapped to this special token for the unknown**. That's **UNK**.

So this **fit_on_texts** will take the text that we have and now create the mapping from each word to an index.

In [ ]:
# Convert to indices.
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='UNK')
tokenizer.fit_on_texts(text)
fragments_indexed = tokenizer.texts_to_sequences(fragments)
targets_indexed = tokenizer.texts_to_sequences(targets)


And then here we are then creating our X and Y tensors that we're going to use to then build and train the model.

In [ ]:


# Convert to appropriate input and output formats.
X = np.array(fragments_indexed, dtype=np.int64)
y = np.zeros((len(targets_indexed), MAX_WORDS))
for i, target_index in enumerate(targets_indexed):
    y[i, target_index] = 1

# Building the Model - Training Process

So let's now look at the model we're going to build.

It's a **sequential model where we're stacking layers on top of each other**.

We will start **with an embedding layer**, and the output dimension. That is basically the **dimensions for the embeddings themselves**. **The input dimensions for the embedding layer is the maximum number of words that we have in our vocabulary**.

we have our **two LSTM layers**.We use return sequences **true** on the first one **because we need the second layer to be able to see all the time steps outputs from this LSTM**. But **we only need the last time step output for the second LSTM**. So there we don't have return sequences equal **true**.

And then we **have a fully connected layer with relu activation**, followed by **a fully connected layer with softmax**. And as usual, with softmax, we use **categorical cross entropy**. And as usual, we use the **adam optimizer**. And then we print out the information about this model and we **start training it**.

And then **split** the data into a validation set that is 5% of the total data set.

In [ ]:
# Build and train model.
training_model = Sequential()
training_model.add(Input(shape=(None,), batch_size=BATCH_SIZE))
training_model.add(Embedding(
    output_dim=EMBEDDING_WIDTH, input_dim=MAX_WORDS,
    mask_zero=True))
training_model.add(LSTM(128, return_sequences=True,
                        dropout=0.2, recurrent_dropout=0.2))
training_model.add(LSTM(128, dropout=0.2,
                        recurrent_dropout=0.2))
training_model.add(Dense(128, activation='relu'))
training_model.add(Dense(MAX_WORDS, activation='softmax'))
training_model.compile(loss='categorical_crossentropy',
                       optimizer='adam')
training_model.summary()
history = training_model.fit(X, y, validation_split=0.05,
                             batch_size=BATCH_SIZE,
                             epochs=EPOCHS, verbose=2,
                             shuffle=True)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (256, None, 100)       │       750,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (256, None, 128)       │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (256, 128)             │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (256, 128)             │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (256, 7500)            │       967,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,982,844 (7.56 MB)

 Trainable params: 1,982,844 (7.56 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/32
94/94 - 90s - 960ms/step - loss: 7.1470 - val_loss: 6.7090
Epoch 2/32
94/94 - 79s - 845ms/step - loss: 6.4303 - val_loss: 6.6338
Epoch 3/32
94/94 - 79s - 838ms/step - loss: 6.3052 - val_loss: 6.6494
Epoch 4/32
94/94 - 80s - 851ms/step - loss: 6.1738 - val_loss: 6.6635
Epoch 5/32
94/94 - 81s - 864ms/step - loss: 6.0340 - val_loss: 6.6750
Epoch 6/32
94/94 - 77s - 822ms/step - loss: 5.9329 - val_loss: 6.7372
Epoch 7/32
94/94 - 81s - 866ms/step - loss: 5.8487 - val_loss: 6.7491
Epoch 8/32
94/94 - 81s - 866ms/step - loss: 5.7728 - val_loss: 6.7928
Epoch 9/32
94/94 - 78s - 832ms/step - loss: 5.6906 - val_loss: 6.8046
Epoch 10/32
94/94 - 80s - 855ms/step - loss: 5.6037 - val_loss: 6.8039
Epoch 11/32
94/94 - 79s - 845ms/step - loss: 5.5111 - val_loss: 6.9695
Epoch 12/32
94/94 - 77s - 821ms/step - loss: 5.4277 - val_loss: 6.9387
Epoch 13/32
94/94 - 78s - 832ms/step - loss: 5.3395 - val_loss: 7.0502
Epoch 14/32
94/94 - 81s - 858ms/step - loss: 5.2646 - val_loss: 7.1323
Epoch 15/32
94/

 this is will take some time to get through all of these 32 epochs. And now the training has completed here with its 32 epochs.

So now we're going to see here if this language model works well. We're going to look at **predicting the next word**, and we will do it with **auto regression**, to **see that it has learned some language structure**

We will present **a number of words to the model**, it'll **predict a next word, and then we will now just take that word and feed it back to the model**.

And in order to do that in **tensorflow**, we need use **stateful model**. we're going to build a **identical model to what we just had, but we will declare the LSTM models here as stateful**.

And what that means is that **once we have done a prediction, they will keep their internal state so we can then just feed another input to it and it'll have the accumulated state from the previous predictions**.

 We have already trained the other model, this one wouldn't have that knowledge.

We can then **take the old model**, **we have the training_model, that's the one we have trained**, and then we just **read out all the weights from that into this weights variable**.

And then we take **this new model, this inference model**, and we **set the weights**. And we can do that **assuming that these two models are completely identical in terms of the number of weights they have,**

The only **difference between these two** is that this **one is declared as stateful while the training model wasn't**.

So we should now be able to **just build this model and initialize it with the weights that we have already trained**.



In [ ]:
# Build stateful model used for prediction.
inference_model = Sequential()
inference_model.add(Input(shape=(None,), batch_size=1))
inference_model.add(Embedding(
    output_dim=EMBEDDING_WIDTH, input_dim=MAX_WORDS,
    mask_zero=False))
inference_model.add(LSTM(128, return_sequences=True,
                         dropout=0.2, recurrent_dropout=0.2,
                         stateful=True))
inference_model.add(LSTM(128, dropout=0.2,
                         recurrent_dropout=0.2, stateful=True))
inference_model.add(Dense(128, activation='relu'))
inference_model.add(Dense(MAX_WORDS, activation='softmax'))
weights = training_model.get_weights()
inference_model.set_weights(weights)

So now let's look at what the **autoregression** will look like when we do this step by step.

So we'll only **pick the highest probability** as we do the next prediction.

**So we'll start with an input sequence.** We'll do two words. So **I saw** is the **input sequence**. Before we call this stateful model, we need **to reset the states because it will retain states from befor**e. So for **the first prediction**, **we want to set the states to zero.** And we do that by calling **reset states**.

 And then the string we have predicted so far, we set that one to **a null string**. So we haven't done any predictions yet.

 And then this first for loop is where we are going to do the, feed these first initial words to the model. We are going through this **first_words_indexed** which is converted to the **tokens**. We will add I saw to it. And then we do the **prediction** on I, and then we do another prediction on saw. Now we have built up an **internal state for this model**.

 And then after that we can move on to start predicting. So this **y_predict** will be what the model **has predicted after we fed it**, both words I saw. And now we can, for a fixed number of future words, we're going to do predictions.

In [ ]:
# Provide beginning of sentence and
# predict next words in a greedy manner
first_words = ['i', 'saw']
first_words_indexed = tokenizer.texts_to_sequences(
    first_words)
inference_model.layers[1].reset_states() #set the states to zero
inference_model.layers[2].reset_states()
predicted_string = ''
# Feed initial words to the model.
for i, word_index in enumerate(first_words_indexed):
    x = np.zeros((1, 1), dtype=np.int64)
    x[0][0] = word_index[0]
    predicted_string += first_words[i]
    predicted_string += ' '
    y_predict = inference_model.predict(x, verbose=0)[0]


So we're going to do this for, **PREDICT_LENGTH** is set to three. So we take out the highest probability from the previous prediction. So, that gives us the **new_word_index** for that we will view as being the **prediction**.

 We will **convert that to its actual string**. So we can then build up the **predicted_string** so we can see what the result was.

 But we'll also use this **new_word_index**, the encoded version of the word as input to** the next time around**. So we create the tensor, only has that **new_word_index**, and t**hen we can do another prediction. So we feed that back to the model. So we do that one step at a time**.

In [ ]:
# Predict PREDICT_LENGTH words.
for i in range(PREDICT_LENGTH):
    new_word_index = np.argmax(y_predict)
    word = tokenizer.sequences_to_texts(
        [[new_word_index]])
    x[0][0] = new_word_index
    predicted_string += word[0]
    predicted_string += ' '
    y_predict = inference_model.predict(x, verbose=0)[0]
print(predicted_string)

i saw the same sky 


And then once we have done that, we **have predicted a number of steps into the future and we just print out the string of words**.

We started with **I saw**, and the model would complete this with **the same sky**.

So, clearly we have a language model that is working.

But we also want to look at what **embeddings it learned**. So if, remember this model, the first stage in the model, the first layer is **this embedding layer that encodes each word into a word vector.**

So let's explore **these word vectors** a little bit.

We will read out the **weights** for **the first layer of the model or the initial layer** and that is our **embeddings**.

We are going to look up the **embeddings for a couple of words**. (ex: of, the)

**for each of these words, we are going to look up what the embedding is**, and **then we'll look up other words, embeddings that are close to that embedding**. And we can see how this embedding layer, how it has chosen to place these different words in vector space.

So what we do is we basically, we go **through all of the embeddings here in this inner loop**, and we **compute the distance between the embedding for the word we have chosen.** **So we have an embedding for the, and we are going to compute the distance to all other embeddings in our vocabulary.**

And then after that, **we will print out the ones that are closest in vector space**.



In [ ]:
# Explore embedding similarities.
embeddings = training_model.layers[0].get_weights()[0]
lookup_words = ['the', 'of']
for lookup_word in lookup_words:
    lookup_word_indexed = tokenizer.texts_to_sequences(
        [lookup_word])
    print('words close to:', lookup_word)
    lookup_embedding = embeddings[lookup_word_indexed[0]]
    word_indices = {}
    # Calculate distances.
    for i, embedding in enumerate(embeddings):
        distance = np.linalg.norm(
            embedding - lookup_embedding)
        word_indices[distance] = i
    # Print sorted by distance.
    for distance in sorted(word_indices.keys())[:5]:
        word_index = word_indices[distance]
        word = tokenizer.sequences_to_texts([[word_index]])[0]
        print(word + ': ', distance)
    print('')

words close to: the
the:  0.0
a:  3.7840977
“the:  3.793902
my:  4.4270105
his:  4.4453144

words close to: of
of:  0.0
in:  0.36596307
with:  0.3759466
mist:  0.4507674
prognosticated:  0.46046245



So we can see here that the** word closest to the is obviously the, because we didn't remove that. So it has a distance of zero**.

But we also see that another word that was quotes and the, so apparently, we didn't really clean the data sufficiently, so this is also viewed as a word and it's makes very much sense that this is a word that is very close to the because it's kind of the same word, But we also see that other words that are close are **things like a**, which makes sense. **They are articles**,



So we see **that the model seems to have found some kind of structure in language that these words are similar to each other, so it makes sense to group them together.**



We haven't really provided information about how a language is built, but by j**ust training it to predict the next word**, it has found this structure in the language itself.